In [1]:
# Google Colab drive mount disabled for local execution
print('Running locally. Using local relative paths.')


Running locally. Using local relative paths.


In [2]:
import os
import pandas as pd
import numpy as np
from collections import defaultdict
import tensorflow as tf
import random

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

I0000 00:00:1781193326.181914   13400 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781193326.184651   13400 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1781193326.290606   13400 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1781193329.289829   13400 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781193329.295321   13400 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [3]:
CROP = "soybean"

# May → October
MONTHS = [5, 6, 7, 8, 9, 10]

print("Crop:", CROP)
print("Months:", MONTHS)

Crop: soybean
Months: [5, 6, 7, 8, 9, 10]


In [4]:
data = np.load("../processed/full_weather_9features_v2.npz", allow_pickle=True)

X_seq_full = data["X_seq_full"]
meta = data["meta"]

print(X_seq_full.shape)

(17927, 12, 12)


In [5]:
# May → October (index 4 to 9)
X_crop = X_seq_full[:, 4:10, :]

print("X_crop shape:", X_crop.shape)

X_crop shape: (17927, 6, 12)


In [6]:
# soybean-specific feature indices
selected_features_idx = [0, 1, 2, 3, 4, 5, 6, 11]

X_crop = X_crop[:, :, selected_features_idx]

print("Updated X_crop shape:", X_crop.shape)

Updated X_crop shape: (17927, 6, 8)


In [7]:
from tensorflow.keras.layers import Input
seq_input = Input(shape=(6, 8))

In [8]:
import pandas as pd
yield_df = pd.read_csv("../processed/usda_soyabean_1.csv")
yield_df.columns = ['fips', 'year', 'yield']
yield_df['fips'] = yield_df['fips'].astype(str).str.zfill(5)
yield_df['year'] = yield_df['year'].astype(str)
yield_df['yield'] = pd.to_numeric(yield_df['yield'], errors='coerce')
yield_df = yield_df.dropna(subset=['yield'])


In [9]:
# ansi processing merged into CSV loader


In [10]:
# grouping merged into CSV loader


In [11]:
print(yield_df.head())
print("Shape:", yield_df.shape)
print("Unique FIPS:", yield_df['fips'].nunique())

    fips  year  yield
0  01009  2019   35.3
1  01019  2019   30.1
2  01033  2019   38.7
3  01043  2019   42.8
4  01047  2019   26.1
Shape: (9147, 3)
Unique FIPS: 1702


In [12]:
final_X = []
final_y = []
final_meta = []

for i, (fips, year) in enumerate(meta):

    match = yield_df[
        (yield_df['fips'] == fips) &
        (yield_df['year'] == year)
    ]

    if len(match) == 1:
        final_X.append(X_crop[i])
        final_y.append(match['yield'].values[0])
        final_meta.append((fips, year))

X = np.array(final_X)
y = np.array(final_y)

print(X.shape, y.shape)

(7449, 6, 8) (7449,)


In [13]:
fips_list = [f for (f, y_) in final_meta]
year_list = [y_ for (f, y_) in final_meta]

fips_arr = np.array(fips_list)
year_arr = np.array(year_list).astype(int)

In [14]:
from collections import defaultdict

yield_map = defaultdict(dict)

# store yield per (fips, year)
for i in range(len(y)):
    yield_map[fips_arr[i]][year_arr[i]] = y[i]

yield_lag_1 = np.zeros(len(y))
yield_lag_2 = np.zeros(len(y))

for i in range(len(y)):
    f = fips_arr[i]
    yr = year_arr[i]

    yield_lag_1[i] = yield_map[f].get(yr - 1, 0)
    yield_lag_2[i] = yield_map[f].get(yr - 2, 0)

yield_trend = yield_lag_1 - yield_lag_2

print("Lag features ready with zero-imputation")


Lag features ready with zero-imputation


In [15]:
# total rainfall
total_rain = X[:, :, 4].sum(axis=1)

# avg temperature
avg_temp_season = X[:, :, 0:2].mean(axis=(1,2))

# flowering rain (July, August)
rain_flowering = X[:, 2:4, 4].sum(axis=1)

# heat stress
heat_stress = (X[:, :, 0] > 35).sum(axis=1)

# total GDD
gdd_total = X[:, :, 3].sum(axis=1)

# avg humidity
avg_humidity = X[:, :, 5].mean(axis=1)

# evapotranspiration total (important!)
et_total = X[:, :, 7].sum(axis=1)


In [16]:
X_extra = np.stack([
    total_rain,
    avg_temp_season,
    yield_trend,
    rain_flowering,
    heat_stress,
    gdd_total,
    avg_humidity,
    et_total
], axis=1)

print("X_extra shape:", X_extra.shape)

X_extra shape: (7449, 8)


In [17]:
from sklearn.preprocessing import LabelEncoder

# Standardize fips_arr as strings to avoid type mixing
fips_arr = np.array([str(int(float(f))).zfill(5) if isinstance(f, (int, float, np.number)) else str(f).zfill(5) for f in fips_arr])
le = LabelEncoder()
fips_encoded = le.fit_transform(fips_arr)


In [18]:
train_mask = year_arr < 2022
test_mask = year_arr == 2022

X_train = X[train_mask]
X_test = X[test_mask]

X_lag_train = X_extra[train_mask]
X_lag_test = X_extra[test_mask]

y_train = y[train_mask]
y_test = y[test_mask]

fips_train = fips_encoded[train_mask]
fips_test = fips_encoded[test_mask]

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (6146, 6, 8)
Test: (1303, 6, 8)


In [19]:
from sklearn.preprocessing import StandardScaler

# sequence scaling
scaler_X = StandardScaler()

X_train_scaled = scaler_X.fit_transform(
    X_train.reshape(-1, X_train.shape[-1])
).reshape(X_train.shape)

X_test_scaled = scaler_X.transform(
    X_test.reshape(-1, X_test.shape[-1])
).reshape(X_test.shape)

# lag scaling
lag_scaler = StandardScaler()
X_lag_train_scaled = lag_scaler.fit_transform(X_lag_train)
X_lag_test_scaled = lag_scaler.transform(X_lag_test)

# target scaling
y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train.reshape(-1,1))
y_test_scaled = y_scaler.transform(y_test.reshape(-1,1))

In [20]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Concatenate, Dropout, Embedding, Flatten
from tensorflow.keras.optimizers import Adam

seq_input = Input(shape=(6, 8))

x = LSTM(128, return_sequences=True)(seq_input)
x = tf.keras.layers.LayerNormalization()(x)

x = LSTM(64)(x)
x = Dropout(0.3)(x)

x = Dense(32, activation='relu')(x)

# lag input
lag_input = Input(shape=(8,))
y_lag = Dense(16, activation='relu')(lag_input)

# fips input
fips_input = Input(shape=(1,))
fips_emb = Embedding(input_dim=3000, output_dim=8)(fips_input)
fips_emb = Flatten()(fips_emb)

# combine
combined = Concatenate()([x, y_lag, fips_emb])

z = Dense(64, activation='relu')(combined)
z = Dropout(0.3)(z)
z = Dense(32, activation='relu')(z)
output = Dense(1)(z)

model = Model(inputs=[seq_input, lag_input, fips_input], outputs=output)

model.compile(
    optimizer=Adam(learning_rate=0.0003),
    loss='mse',
    metrics=['mae']
)

model.summary()

E0000 00:00:1781193377.904426   13400 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 6, 8)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 6, 128)    │     70,144 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 6, 128)    │        256 │ lstm[0][0]        │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 64)        │     49,408 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_3       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 64)        │          0 │ lstm_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_2       │ (None, 8)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 1, 8)      │     24,000 │ input_layer_3[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32)        │      2,080 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 16)        │        144 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 8)         │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 56)        │          0 │ dense[0][0],      │
│ (Concatenate)       │                   │            │ dense_1[0][0],    │
│                     │                   │            │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │      3,648 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 32)        │      2,080 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 1)         │         33 │ dense_3[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 151,793 (592.94 KB)

 Trainable params: 151,793 (592.94 KB)

 Non-trainable params: 0 (0.00 B)

In [21]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    [X_train_scaled, X_lag_train_scaled, fips_train], y_train_scaled,
    validation_data=([X_test_scaled, X_lag_test_scaled, fips_test], y_test_scaled),
    epochs=50,
    batch_size=64,
    callbacks=[early_stop],
    shuffle=False
)

Epoch 1/50


 1/97 ━━━━━━━━━━━━━━━━━━━━ 11:37 7s/step - loss: 2.5939 - mae: 1.4109

 4/97 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 1.7846 - mae: 1.0836 

 7/97 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - loss: 1.5685 - mae: 1.0054

10/97 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 1.4197 - mae: 0.9540

...

90/97 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 0.1097 - mae: 0.2518

92/97 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 0.1097 - mae: 0.2519

94/97 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 0.1097 - mae: 0.2519

96/97 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 0.1097 - mae: 0.2520

97/97 ━━━━━━━━━━━━━━━━━━━━ 3s 34ms/step - loss: 0.1126 - mae: 0.2568 - val_loss: 0.3741 - val_mae: 0.4863


In [22]:
from sklearn.metrics import r2_score

y_pred_scaled = model.predict([X_test_scaled, X_lag_test_scaled, fips_test])
y_pred = y_scaler.inverse_transform(y_pred_scaled)

r2 = r2_score(y_test, y_pred)

print("🔥 Soybean R²:", r2)

 1/41 ━━━━━━━━━━━━━━━━━━━━ 23s 589ms/step

 8/41 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step   

15/41 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step

20/41 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step

27/41 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step

32/41 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step

37/41 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step

41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step

41/41 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step


🔥 Soybean R²: 0.7360457767724957


In [23]:
model.save("../Soybean_lstm_model.keras")

In [24]:
model.save("../Soybean_lstm_model_v2.keras")

In [25]:
import joblib

joblib.dump(scaler_X, "../soybean_x_scaler.pkl")
joblib.dump(lag_scaler, "../soybean_lag_scaler.pkl")
joblib.dump(y_scaler, "../soybean_y_scaler.pkl")
joblib.dump(le, "../soybean_fips_encoder.pkl")

['../soybean_fips_encoder.pkl']

In [26]:
import pandas as pd

history_df = pd.DataFrame(history.history)
history_df.to_csv('../Soyabean_model_training_history.csv', index=False)
print('Training history saved successfully.')

Training history saved successfully.


In [27]:
import json

config = {
    "crop": "soybean",

    "months_used": [5, 6, 7, 8, 9, 10],

    "sequence_features": [
        {"index": 0, "name": "max_temp_C"},
        {"index": 1, "name": "min_temp_C"},
        {"index": 2, "name": "temp_range"},
        {"index": 3, "name": "gdd"},
        {"index": 4, "name": "precipitation"},
        {"index": 5, "name": "relative_humidity"},
        {"index": 6, "name": "radiation"},
        {"index": 7, "name": "evapotranspiration"}
    ],

    "engineered_features": {
        "total_rain": "sum(X[:, :, 4])",
        "avg_temp_season": "mean(X[:, :, 0:2])",
        "rain_flowering": "sum(X[:, 2:4, 4])",
        "heat_stress": "count(X[:, :, 0] > 35)",
        "gdd_total": "sum(X[:, :, 3])",
        "avg_humidity": "mean(X[:, :, 5])",
        "et_total": "sum(X[:, :, 7])"
    },

    "lag_features": [
        "yield_lag_1",
        "yield_lag_2",
        "yield_trend",
        "total_rain",
        "avg_temp_season",
        "rain_flowering",
        "heat_stress",
        "gdd_total",
        "avg_humidity",
        "et_total"
    ],

    "input_shape": {
        "sequence": [6, 8],
        "lag": 8
    },

    "notes": "GDD computed as avg_temp_C - 10 (Celsius). Feature indices updated after selection."
}

with open('../soybean_config.json', 'w') as f:
    json.dump(config, f, indent=4)

print("Soybean config saved successfully.")

Soybean config saved successfully.


In [28]:
np.savez("../soybean_data.npz",
         X=X,
         X_extra=X_extra,
         y=y,
         fips=fips_arr,
         year=year_arr)